```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0 done;
    class A1a current;
    class A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 01: Corpus Metadata Exploration
## Understanding Project Gutenberg's Philosophy Collection

**Goal:** Explore the Project Gutenberg catalog to understand what philosophy texts are available, make informed decisions about corpus construction, and generate a curated list of books to download.

**What we will do:**
1. Load the official Project Gutenberg catalog
2. Identify philosophy-related books
3. Analyse languages
4. Analyse subjects
5. Analyse authors
6. Make selection decisions
7. Extract more metadata: Wikipedia URLs
8. Coprus temporal analysis
9. Save a filtered list for text downloading

# Setup

In [ ]:
!pip install matplotlib seaborn tqdm rdflib ipywidgets

In [ ]:
# -----------------------------
# Import
# -----------------------------

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
import tarfile
from pathlib import Path
from collections import Counter
from rdflib import Graph, Namespace

from tqdm.auto import tqdm

# -----------------------------
# Parameters
# -----------------------------
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("\n✓ Libraries loaded")

# Part 1: Load Project Gutenberg Catalog

Download the catalog from: https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv

Save the file in the folder NLP2026/Notebooks, where this notebook has been saved.

This file contains metadata for all ~70,000 books in Project Gutenberg.

In [ ]:
# Configuration
CATALOG_PATH = './analysis/tables/pg_catalog.csv'
RDF_CATALOG_PATH = './data/pg_rdf-files.tar.bz2'
OUTPUT_DIR = Path('./analysis')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"\nCatalog path: {CATALOG_PATH}")
print(f"RDF catalog path: {RDF_CATALOG_PATH}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

In [ ]:
# Load the catalog
catalog = pd.read_csv(CATALOG_PATH, low_memory=False)

# Lowercase columns' names for consistency
catalog.columns = catalog.columns.str.lower()

print(f"\n✓ Loaded catalog with {len(catalog):,} books")
print(f"\nColumns available:")
for col in catalog.columns:
    print(f"  - {col}")

In [ ]:
# Remove confusing characters from columns' names (#)
catalog = catalog.rename(columns={'text#': 'pg_id'})

# Fill the gap

### _Check what the columns contain_

- What do the columns contain? 

- What are **Subjects**? 

- What is **LoCC**? 

- What are **Bookshelves**? 

- What is **Issued**? 

- Some cells containt the word `NaN`, what does it mean?

In [ ]:
# Display sample rows to understand the data
print("\nSample of the catalog:\n")
# =============================================== YOUR CODE HERE ===============================================


# Part 2: Identify Philosophy Books

Let's find all books related to philosophy by searching in subject/category fields.

**Note:** Project Gutenberg stores subject information in multiple fields:
- **Subjects**: Library of Congress Subject Headings
- **Bookshelves**: Project Gutenberg curated categories
- **LoCC**: Library of Congress Classification codes

In [ ]:
# Function to search for keyword in subjects
def search_subjects(df: pd.DataFrame, keyword: str, case_sensitive=False) -> pd.DataFrame:
    """
    Search for books containing a keyword in subjects OR Bookshelves fields.
    Returns a filtered dataframe.
    """
    if case_sensitive:
        mask_subjects = df['subjects'].fillna('').str.contains(keyword, na=False)
        mask_bookshelves = df['bookshelves'].fillna('').str.contains(keyword, na=False)
    else:
        mask_subjects = df['subjects'].fillna('').str.contains(keyword, case=False, na=False)
        mask_bookshelves = df['bookshelves'].fillna('').str.contains(keyword, case=False, na=False)
    
    # Combine the matches from either column
    mask = mask_subjects | mask_bookshelves
    
    return df[mask]

# Fill the gap

### _How to find all the books about **philosophy**_

In [ ]:
# Complete the line below to find all books with "philosophy" in subjects or bookshelves using the function
# `search_subjects` defined above to create a dataframe that contains only these philosophy books
# =============================================== YOUR CODE HERE ===============================================
philosophy_books = # Use `search_subjects` to create a datafram with only the philosophy books

print(f"Found {len(philosophy_books):,} books containing 'philosophy' in subjects or bookshelves")
print(f"This is {len(philosophy_books)/len(catalog)*100:.2f}% of the total catalog")

In [ ]:
# Sample of philosophy books
print("Sample philosophy books:")
philosophy_books[['pg_id', 'title', 'authors', 'subjects', 'bookshelves', 'language']].head(20)

# Critical thinking

- Does the selection make sense? 

- Are there titles that should not be in the selection?

- Are there missing titles?

 =============================================== YOUR THOUGHTS HERE ===============================================



---

# Explore Subject Keywords

Let's see what specific philosophy subjects exist in the catalog.

In [ ]:
# Extract all unique subjects from philosophy books
all_subjects = []
for subjects in philosophy_books['subjects'].dropna():
    # Subjects are often separated by semicolons
    subjects_list = [s.strip() for s in str(subjects).split(';')]
    all_subjects.extend(subjects_list)

subject_counts = Counter(all_subjects)

print(f"Found {len(subject_counts)} unique subject tags in philosophy books\n")
print("Top 30 most common subjects:")
print("=" * 80)
for subject, count in subject_counts.most_common(30):
    print(f"{count:4d} | {subject}")

# Fill the gap

### _Display bookshelves keywords_

Use the code in the cell above to loop over the dataframe, but this time we want to extract **Bookshelves** keywords, not Subjects.

In [ ]:
# Extract all unique Bookshelves keywords from philosophy books following the example above
# =============================================== YOUR CODE HERE ===============================================
all_bookshelves = []
for shelf in                                      # Adapt the loop from the cell above to this 
    shelf_list =                                  # Complete the list comprehension 
    all_bookshelves.extend(shelf_list)

bookshelf_counts = Counter(all_bookshelves)

print(f"\nTop 20 most common bookshelves:")
print("=" * 80)
for shelf, count in bookshelf_counts.most_common(20):
    print(f"{count:4d} | {shelf}")

In [ ]:
# Function to explore related keywords
def explore_keyword(keyword):
    """
    Find all subjects and bookshelves containing a specific keyword.
    Useful for exploring related terms like 'ethics', 'logic', 'political', etc.
    """
    # Search in subjects
    matching_subjects = [s for s in subject_counts.keys() if keyword.lower() in s.lower()]
    print(f"\nSubjects containing '{keyword}' ({len(matching_subjects)} found):")
    print("=" * 80)
    for subj in sorted(matching_subjects):
        print(f"{subject_counts[subj]:4d} | {subj}")
    
    # Search in bookshelves
    matching_shelves = [s for s in bookshelf_counts.keys() if keyword.lower() in s.lower()]
    print(f"\nBookshelves containing '{keyword}' ({len(matching_shelves)} found):")
    print("=" * 80)
    for shelf in sorted(matching_shelves):
        print(f"{bookshelf_counts[shelf]:4d} | {shelf}")
    
    # return matching_subjects, matching_shelves

# Fill the gap

### _Explore different keywords_

Use the function `explore_keyword` defined above to explore different relevant keywords

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Display the 'Subjects' containing the keyword "philosophy"


In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Display the 'Subjects' containing the keyword "logic"


In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Display the 'Subjects' containing the keyword "ethics"


In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Display the 'Subjects' containing the keyword...


In [ ]:
# Try other keywords to see if we're missing anything
# Uncomment and run these to explore:



In [ ]:
# Try other keywords to see if we're missing anything
# Uncomment and run these to explore:



In [ ]:
# Try other keywords to see if we're missing anything
# Uncomment and run these to explore:



In [ ]:
# Try other keywords to see if we're missing anything
# Uncomment and run these to explore:



In [ ]:
# Try other keywords to see if we're missing anything
# Uncomment and run these to explore:


# Part 3: Language Analysis

What languages are represented? What bias does this introduce?

In [ ]:
# Language distribution in philosophy books
language_counts = philosophy_books['language'].value_counts()

print(f"Philosophy books by language:")
print("=" * 60)
for lang, count in language_counts.head(20).items():
    percentage = count / len(philosophy_books) * 100
    bar = '█' * int(percentage / 2)
    print(f"{lang:10} ({count:4d}, {percentage:5.1f}%) {bar}")

print(f"\n{language_counts.get('en', 0) / len(philosophy_books) * 100:.1f}% of the selected philosophy books are in English.")

In [ ]:
# Visualize language distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Bar chart of top 10 languages
ax1 = axes[0]
top_languages = language_counts.head(10)
top_languages.plot(kind='bar', ax=ax1, color='teal')
ax1.set_title('Top 10 languages in Gutenberg philosophy books')
ax1.set_xlabel('Language Code')
ax1.set_ylabel('Number of Books')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3, axis='y')

# Right: Pie chart showing English vs Others
ax2 = axes[1]
english_count = language_counts.get('en', 0)
others_count = len(philosophy_books) - english_count
ax2.pie([english_count, others_count], 
        labels=['English', 'Other Languages'],
        autopct='%1.1f%%',
        colors=['teal', 'goldenrod'],
        startangle=90)
ax2.set_title('English vs other languages in the corpus')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb01-language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Critical thinking

- What philosophical traditions are excluded by focusing on English?
- Are we studying 'philosophy' or 'Anglophone philosophy'?
- What biases exist in translation selection?

 =============================================== YOUR THOUGHTS HERE ===============================================



---

# Part 4: Subject distribution

What types of philosophy are represented?

In [ ]:
# Define rules as a list of (keywords, label) pairs
# Add/remove/edit as needed.
CATEGORIES = [
    (['ethics', 'moral'], 'Ethics'),
    (['political'], 'Political Philosophy'),
    (['logic'], 'Logic'),
    (['metaphysics', 'ontology'], 'Metaphysics'),
    (['epistemology', 'knowledge'], 'Epistemology'),
    (['aesthetics', 'beauty'], 'Aesthetics'),
    (['religion', 'theology'], 'Philosophy of Religion'),
    (['science'], 'Philosophy of Science'),
]

# Optional: default label if no rule matches
DEFAULT_CATEGORY = 'General/Other'

In [ ]:
# Categorise philosophy subfields
# Note: These are approximations based on keywords

def categorise_philosophy(row: pd.Series, rules:list = CATEGORIES, default:str = DEFAULT_CATEGORY) -> str:
    """
    Categorize books into philosophy subfields based on subjects and bookshelves.
    
    Parameters:
        row : pandas Series (must contain the columns 'subjects' and 'bookshelves')
        rules : list of (keywords, label) tuples
        default : str, returned if no rule matches
    
    Returns:
        str: semicolon-separated list of matching categories
    """
    subjects = str(row.get('subjects', '')).lower()
    shelves = str(row.get('bookshelves', '')).lower()
    combined = subjects + ' ' + shelves
    
    categories = []
    for keywords, label in rules:
        if any(keyword in combined for keyword in keywords):
            categories.append(label)
    
    if not categories:
        categories.append(default)
    
    return '; '.join(categories)

In [ ]:
philosophy_books['category'] = philosophy_books.apply(categorise_philosophy, axis=1)

# Count categories (books can belong to multiple)
all_categories = []
for cat_string in philosophy_books['category']:
    all_categories.extend([c.strip() for c in cat_string.split(';')])

category_counts = Counter(all_categories)

print("Distribution by philosophy subfield:")
print("=" * 60)
for category, count in category_counts.most_common():
    percentage = count / len(philosophy_books) * 100
    bar = '█' * int(percentage / 3)
    print(f"{category:25} ({count:4d}, {percentage:5.1f}%) {bar}")

In [ ]:
# Visualise subfields distribution
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

category_series = pd.Series(dict(category_counts.most_common()))
category_series.plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Number of books')
ax.set_title('Philosophy subfields distribution')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb01-subfield_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved under {OUTPUT_DIR}/figures/nb01-subfield_distribution.png")

# Part 5: Author analysis

Which philosophers are most represented?

In [ ]:
def get_primary_author(author_string: str) -> str:
    """
    Extract the first author from a semicolon-separated string with authors' names.

    This function is designed to be used with pandas `.apply()` on a Series
    containing author information from Project Gutenberg metadata.

    The function:
    1. Handles missing values (NaN, empty strings) by returning 'Unknown'.
    2. Splits the string by semicolon and takes the first author.
    3. Removes trailing information like dates (e.g., ', 1875') and translator
       metadata, leaving only the primary author's name.

    Parameters
    ----------
    author_string : str
        A single value from the 'Authors' column. This can be:
        - A semicolon-separated string of authors (e.g., "Smith, John; Jones, Mary")
        - An empty string ('')
        - pandas NaN (missing value)
        - None

    Returns
    -------
    str
        The name of the primary author, cleaned of date/translator suffixes.
        If the input is missing or empty, returns 'Unknown'.

    Notes
    -----
    - The function removes any content after the first comma that is followed
      by a 3- or 4-digit year (e.g., ', 1806-1873' or ', 1875').
    - It does not remove other types of suffixes (e.g., ';' or ':' are handled
      by the semicolon split and the date removal only).
    - The regex pattern `r',\\s*\\d{3,4}.*'` matches:
        - a comma,
        - optional whitespace,
        - a 3- or 4-digit number (year),
        - and everything after it.
    """
    if pd.isna(author_string) or author_string == '':
        return 'Unknown'

    # Split by semicolon and take the first author
    authors = str(author_string).split(';')
    primary = authors[0].strip()

    # Remove dates and translator info (e.g., ', 1806-1873' or ', 1875')
    primary = re.sub(r',\s*\d{3,4}.*', '', primary)

    return primary

#### How to use and example output:
    --------
    >>> get_primary_author("Kant, Immanuel; Mendelssohn, Moses")
    'Kant, Immanuel'

    >>> get_primary_author("Mill, John Stuart, 1806-1873; Editor: Smith")
    'Mill, John Stuart'

    >>> get_primary_author("Aristotle; Translator: Ross")
    'Aristotle'

    >>> get_primary_author(None)
    'Unknown'

    >>> get_primary_author("")
    'Unknown'

## How `get_primary_author` processes different input into similar outputs:


| Input (`Authors`) | Output (`primary_author`) |
| :--- | :--- |
| Kant, Immanuel; Mendelssohn, Moses | Kant, Immanuel |
| Mill, John Stuart, 1806-1873 | Mill, John Stuart |
| Aristotle; Translator: Ross | Aristotle |
| Plato, 428-348 BC; Translator: Jowett | Plato |
| NaN | Unknown |
| "" (empty string) | Unknown |
| Descartes, René, 1596-1650 | Descartes, René |

In [ ]:
philosophy_books['primary_author'] = philosophy_books['authors'].apply(get_primary_author)

author_counts = philosophy_books['primary_author'].value_counts()

print(f"Total unique authors: {len(author_counts)}\n")
print("Top 30 most represented authors:")
print("=" * 60)
for author, count in author_counts.head(30).items():
    print(f"{count:3d} | {author}")

In [ ]:
# Check for the philosophers whom we should include in our corpus
philosophers = [
    'Plato', 'Aristotle', 'Descartes', 'Spinoza', 'Leibniz', 'Locke', 
    'Hume', 'Kant', 'Hegel', 'Nietzsche', 'Mill', 'Rousseau',
    'Berkeley', 'Hobbes', 'Schopenhauer'
] # Name can be added to complete the list

print("\nCanonical figure check:")
print("=" * 60)
for philosopher in philosophers:
    # Search for philosopher in author names
    matches = philosophy_books[philosophy_books['primary_author'].str.contains(philosopher, case=False, na=False)]
    if len(matches) > 0:
        print(f"✓ {philosopher:15} - {len(matches):2d} books")
    else:
        print(f"✗ {philosopher:15} - NOT FOUND")

---
# Critical thinking:
- Which important philosophers are missing?
- Does the corpus over-represent certain traditions?
- Are women philosophers adequately represented?

 =============================================== YOUR THOUGHTS HERE ===============================================



---

# Part 6: Selection criteria

# Fill the gap

### _Check the selection criteria based on the exploration above_

**Instructions:**
1. Review the subject distribution, language analysis, and author representation
2. Decide what to include/exclude for your research question
3. Fill in the criteria below
4. Run the cell to apply filters

**Example for studying modernity in philosophy (1500-1900):**
- Focus on English-language texts (acknowledge translation bias)
- Include core philosophy subjects
- Exclude fiction, poetry misclassified as philosophy
- No specific length restrictions (we'll check this after)

In [ ]:
# Complete the code below to fill in your selection criteria

# =============================================== YOUR CODE HERE ===============================================

SELECTION_CRITERIA = {
    # Languages to include (e.g., ['en'] for English only, or ['en', 'fr', 'de'])
    'languages': [],

    # Subjects and shelves' mark to include
    'subjects_included': [],
    
    # Subjects to EXCLUDE 
    'subjects_excluded': [],

    # Types to EXCLUDE 
    'types_excluded': ['Sound'],
    
    # Specific authors to exclude
    'authors_excluded': [],
    
    # Minimum/maximum text length in characters (None for no limit)
    'min_length': None,
    'max_length': None,
}

print("Selection criteria defined:")
for key, value in SELECTION_CRITERIA.items():
    print(f"  {key}: {value}")

### Apply Selection Criteria

#### _Narrow selection: subjects only_

In [ ]:
# Start with philosophy books
df_selected = philosophy_books.copy()

# Apply language filter
if SELECTION_CRITERIA['languages']:
    df_selected = df_selected[df_selected['language'].isin(SELECTION_CRITERIA['languages'])]
    print(f"After language filter: {len(df_selected)} books")

# Apply subject exclusions
if SELECTION_CRITERIA['subjects_excluded']:
    for subjects_excluded in SELECTION_CRITERIA['subjects_excluded']:
        mask = ~df_selected['subjects'].fillna('').str.contains(subjects_excluded, case=False, na=False)
        df_selected = df_selected[mask]
    print(f"After excluding subjects: {len(df_selected)} books")

# Apply subject inclusions (if specified)
if SELECTION_CRITERIA['subjects_included']:
    mask = df_selected['subjects'].fillna('').str.contains('|'.join(SELECTION_CRITERIA['subjects_included']), case=False, na=False)
    df_selected = df_selected[mask]
    print(f"After including only specified subjects: {len(df_selected)} books")

# Apply types exclusions
if SELECTION_CRITERIA['types_excluded']:
    for types_excluded in SELECTION_CRITERIA['types_excluded']:
        mask = ~df_selected['type'].fillna('').str.contains(types_excluded, case=False, na=False)
        df_selected = df_selected[mask]
    print(f"After excluding specified types: {len(df_selected)} books")

#### _Large selection: subjects and bookshleves_

In [ ]:
# Start with philosophy books
df_selected_large = philosophy_books.copy()

print(f"Starting with: {len(df_selected_large):,} philosophy books")
print()

# Filter by language
if SELECTION_CRITERIA['languages']:
    df_selected_large = df_selected_large[df_selected_large['language'].isin(SELECTION_CRITERIA['languages'])]
    print(f"After language filter: {len(df_selected_large):,} books")

# Use the columns 'Bookshleves' and 'Subjects' to obtain more titles
if SELECTION_CRITERIA['subjects_included']: 
    pattern = '|'.join(SELECTION_CRITERIA['subjects_included'])
    mask_subjects = df_selected_large['subjects'].fillna('').str.contains(pattern, case=False, na=False) # Subjects
    mask_bookshelves = df_selected_large['bookshelves'].fillna('').str.contains(pattern, case=False, na=False) # Bookshelves
    df_selected_large = df_selected_large[mask_subjects | mask_bookshelves]
    print(f"After subject filter: {len(df_selected_large):,} books")

# Exclude specific subjects
if SELECTION_CRITERIA['subjects_excluded']:
    for subjects_excluded in SELECTION_CRITERIA['subjects_excluded']:
        before = len(df_selected_large)
        mask = df_selected_large['subjects'].fillna('').str.contains(subjects_excluded, case=False, na=False)
        df_selected_large = df_selected_large[~mask]
        removed = before - len(df_selected_large)
        print(f"Excluded '{subjects_excluded}': removed {removed} books, {len(df_selected_large):,} remaining")

# Exclude specific authors
if SELECTION_CRITERIA['authors_excluded']:
    for authors_excluded in SELECTION_CRITERIA['authors_excluded']:
        before = len(df_selected_large)
        mask = df_selected_large['authors'].fillna('').str.contains(authors_excluded, case=False, na=False)
        df_selected_large = df_selected_large[~mask]
        removed = before - len(df_selected_large)
        print(f"Excluded author '{authors_excluded}': removed {removed} books, {len(df_selected_large):,} remaining")

# Apply types exclusions
if SELECTION_CRITERIA['types_excluded']:
    for types_excluded in SELECTION_CRITERIA['types_excluded']:
        before = len(df_selected_large)
        mask = ~df_selected_large['type'].fillna('').str.contains(types_excluded, case=False, na=False)
        df_selected_large = df_selected_large[mask]
        removed = before - len(df_selected_large)
        print(f"Excluded types '{types_excluded}': removed {removed} books, {len(df_selected_large):,} remaining")

print()
print("=" * 60)
print(f"✓ FINAL SELECTION: {len(df_selected_large):,} books")
print("=" * 60)

# Critical thinking

- What is the difference between the narrow and the large selection?

---

 =============================================== YOUR THOUGHTS HERE ===============================================



---

In [ ]:
# Build comparison data
comparison_data = []

for keyword in SELECTION_CRITERIA['subjects_included']:
    # Count for df_selected (subjects only)
    count1 = df_selected['subjects'].fillna('').str.contains(keyword, case=False).sum()
    
    # Count for df_selected_large (Subjects + Bookshelves)
    count2 = df_selected_large['subjects'].fillna('').str.contains(keyword, case=False).sum()
    count2 += df_selected_large['bookshelves'].fillna('').str.contains(keyword, case=False).sum()
    
    comparison_data.append({
        'Keyword': keyword,
        'df_selected (S)': count1,
        'df_selected_large (S+B)': count2,
        'Difference': count2 - count1
    })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display
print("\nKeyword match comparison")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)
print(f"Total df_selected: {len(df_selected)} books")
print(f"Total df_selected_large: {len(df_selected_large)} books")

In [ ]:
# Show sample of selected books
print("Sample of selected books:")
df_selected[['pg_id', 'title', 'authors', 'subjects', 'language']].head(20)

# Critical thinking 

- What is missing?
- How will our corpus affect our analysis?


 =============================================== YOUR THOUGHTS HERE ===============================================



---

## Extract Additional Metadata from RDF Files

**Where can we find the relevant information:**
pg_catalog.csv does not contain authors' birth and deathdate. We will use therefore RDF files.

### _What are RDF files?_
**RDF (Resource Description Framework)** is a standardized format for storing structured metadata. Think of it as a way to describe relationships between things in a machine-readable format.

**How RDF works:**
RDF stores information as "triples" - statements with three parts:
- **Subject:** The thing being described (e.g., "Book #4280")
- **Predicate:** The property or relationship (e.g., "has author")
- **Object:** The value (e.g., "Immanuel Kant")

**Why the Project Gutenberg uses RDF:**
- **Machine-readable:** Programs can automatically extract and process metadata
- **Standardized vocabularies:** Uses common standards like Dublin Core and MARC
- **Linked data:** Can connect to other databases (Wikipedia, library catalogs, etc.)

The RDF files of the Project Gutenberg can be downloaded here: https://www.gutenberg.org/cache/epub/feeds/

**In our project:**
We're using Project Gutenberg's RDF files (one per book, ~70,000 total) to extract metadata that is not available in the simpler CSV catalog - specifically author dates and Wikipedia links.

For the selected books, we'll extract:
1. **Wikipedia URLs** (for manual date lookup)
2. **Author birth/death dates** (for temporal analysis)

**Estimated time:** ~2-5 minutes for 500 books

# Part 7: Extract Wikipedia URLs

Many books in the Project Gutenberg have associated Wikipedia pages that contain detailed bibliographic information, including **original publication dates**.

The RDF metadata files include links to Wikipedia pages (when available) under the `webpage` field. We'll extract these URLs for our selected books to:
- Facilitate manual research for publication dates
- Provide students with authoritative sources for additional context
- Create a structured workflow for the manual date lookup task

**What to expect:**
- Not all books will have Wikipedia links (~70-80% coverage typically)
- Some Wikipedia links point to author pages rather than specific book pages
- Links may be in different languages (en.wikipedia, de.wikipedia, etc.)

In [ ]:
def extract_wikipedia_urls(
    book_ids: list[int],
    catalog_path: str
) -> dict[int, str]:
    """
    Extract Wikipedia URLs from Project Gutenberg RDF metadata files.

    The RDF files are contained in a tarball (named 'pg_catalog.tar.bz2').
    For each book ID, the function looks for a 'webpage' predicate whose object
    contains the string 'wikipedia', and returns that URL. If no such URL is found,
    the entry is set to None.

    Parameters
    ----------
    book_ids : list of integers
        A collection of Gutenberg book IDs (e.g., [10108, 10112, ...]).
        They are integers and will be converted to strings internally.
    catalog_path : str
        Path to the RDF catalog tarball (usually 'pg_catalog.tar.bz2').
        The file must be in bzip2-compressed tar format.

    Returns
    -------
    Dict[int or str, Optional[str]]
        A dictionary where each key is a book ID (int) and the value
        is either the Wikipedia URL (as a string) or None if no URL was found or
        an error occurred during extraction.

    Notes
    -----
    - The function assumes that the RDF file inside the tarball follows the path
      pattern: 'cache/epub/{book_id}/pg{book_id}.rdf'.
    - The predicate used to locate the URL is any predicate whose string representation
      contains the word 'webpage' (case-insensitive).
    - If multiple 'webpage' links exist, only the first one that contains 'wikipedia'
      is returned.
    - Errors (e.g., missing RDF file, parsing errors) are caught and result in None.
    - The function uses `tqdm` to display a progress bar.

    Example output
    -------
    >>> urls = extract_wikipedia_urls([10108, 10112], 'data/pg_catalog.tar.bz2')
    >>> print(urls)
    {10108: 'https://en.wikipedia.org/wiki/...', 10112: None}
    """
    wikipedia_urls = {}
    # Convert book_ids to list to allow iteration and reuse
    book_ids_list = list(book_ids)

    with tarfile.open(catalog_path, 'r:bz2') as tar:
        for book_id in tqdm(book_ids_list, desc="Extracting Wikipedia URLs"):
            # Ensure book_id is a string for path construction
            book_id_str = str(book_id)
            rdf_path = f'cache/epub/{book_id_str}/pg{book_id_str}.rdf'

            try:
                member = tar.getmember(rdf_path)
                rdf_file = tar.extractfile(member)

                g = Graph()
                g.parse(rdf_file, format='xml')

                # Search for a webpage predicate pointing to Wikipedia
                found_url = None
                for s, p, o in g:
                    if 'webpage' in str(p).lower():
                        url = str(o)
                        if 'wikipedia' in url.lower():
                            found_url = url
                            break

                wikipedia_urls[book_id] = found_url

            except Exception:
                wikipedia_urls[book_id] = None

    return wikipedia_urls

In [ ]:
# Extract Wikipedia URLs
selected_ids = df_selected['pg_id'].tolist()
print(f"Extracting Wikipedia URLs for {len(selected_ids)} selected books...\n")
wikipedia_urls = extract_wikipedia_urls(selected_ids, RDF_CATALOG_PATH)

# Add to dataframe
df_selected['wikipedia_url'] = df_selected['pg_id'].map(wikipedia_urls)

# Report coverage
has_wikipedia = df_selected['wikipedia_url'].notna().sum()
print(f"\nWikipedia URLs found: {has_wikipedia} / {len(df_selected)} ({has_wikipedia/len(df_selected)*100:.1f}%)")

In [ ]:
def print_empty_cells(df: pd.DataFrame, column: str, show_rows: int = 50) -> pd.DataFrame:
    """
    Print (and return) rows where df[column] is empty:
    - NA/NaN
    - empty string ""
    - whitespace-only strings
    """
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found. Available: {list(df.columns)}")

    s = df[column]

    is_empty = s.isna()
    if s.dtype == "object" or pd.api.types.is_string_dtype(s):
        is_empty = is_empty | s.astype(str).str.strip().eq("")

    empty_rows = df.loc[is_empty].copy()

    print(f"Empty cells in '{column}': {len(empty_rows)} / {len(df)}")
    if len(empty_rows):
        display(empty_rows.head(show_rows))
    return empty_rows

In [ ]:
empty_wiki_url = print_empty_cells(df_selected, "wikipedia_url", 10)

# Part 8: Corpus temporal analysis: extract author birth/death dates (temporal proxy)

Since original publication dates are not available in the RDF metadata, we will use first **author death dates as a temporal proxy** to understand the rough periodization of our corpus.

**Important limitations:**
- This tells us when the *author died*, not when the *work was published*
- A philosopher who died in 1650 may have written works across decades (1610-1650)
- Posthumous publications will not be accurately dated
- Some ancient authors have no precise dates

**Why we are doing this:**
- It gives us a rough sense of corpus coverage (Ancient? Medieval? Early Modern? Modern?)
- Helps assess whether we have enough material for studying modernity (1500-1900)
- Useful for preliminary analysis before we complete manual publication date research

**Technical note:** 
Example triples for a philosophy book:
The RDF files list translators before authors, so we extract the *last* birth/death dates to get the primary author (not translator).

**Limitations:**
- A text written in 1641 by Descartes (died 1650) appears as "1650"
- Posthumous publications, collected works, and late-life writings are conflated
- This is good enough for broad periodization (Ancient, Medieval, Early Modern, Modern)
- NOT precise enough for decade-by-decade analysis

**For precise temporal analysis:** only manual look up of publication dates will give us more precise dates.

In [ ]:
def extract_author_dates_from_rdf(book_ids: list[int], catalog_path: str) -> dict:
    """
    Extract PRIMARY AUTHOR birth and death dates from RDF files.
    Follows the 'creator' relationship to find author and avoid translators/editors.
    """
    from rdflib import Namespace
    
    author_dates = {}
    DCTERMS = Namespace("http://purl.org/dc/terms/")
    PGTERMS = Namespace("http://www.gutenberg.org/2009/pgterms/")
    
    with tarfile.open(catalog_path, 'r:bz2') as tar:
        for book_id in tqdm(book_ids, desc="Extracting author dates"):
            rdf_path = f'cache/epub/{book_id}/pg{book_id}.rdf'
            
            try:
                member = tar.getmember(rdf_path)
                rdf_file = tar.extractfile(member)
                
                g = Graph()
                g.parse(rdf_file, format='xml')
                
                # Find the creator (author) agent URI
                creator_agent = None
                for s, p, o in g:
                    if 'creator' in str(p):
                        creator_agent = o
                        break
                
                # Get birthdate/deathdate for that specific agent
                birthdate = None
                deathdate = None
                
                if creator_agent:
                    for s, p, o in g:
                        if s == creator_agent:  # Only for creator agent
                            predicate = str(p).split('/')[-1].split('#')[-1]
                            if predicate == 'birthdate':
                                birthdate = str(o)
                            elif predicate == 'deathdate':
                                deathdate = str(o)
                
                author_dates[book_id] = {
                    'birthdate': birthdate,
                    'deathdate': deathdate
                }
                
            except Exception as e:
                author_dates[book_id] = {'birthdate': None, 'deathdate': None}
    
    return author_dates

In [ ]:
# Extract author dates
print(f"\nExtracting author dates for {len(selected_ids)} selected books...\n")
author_dates = extract_author_dates_from_rdf(selected_ids, RDF_CATALOG_PATH)

# Add to dataframe
df_selected['author_birthdate'] = df_selected['pg_id'].map(lambda x: author_dates.get(x, {}).get('birthdate'))
df_selected['author_deathdate'] = df_selected['pg_id'].map(lambda x: author_dates.get(x, {}).get('deathdate'))

# Report coverage
has_deathdate = df_selected['author_deathdate'].notna().sum()
print(f"\n✓ Author death dates found: {has_deathdate} / {len(df_selected)} ({has_deathdate/len(df_selected)*100:.1f}%)")

In [ ]:
# Show sample with new metadata
print("\nSample of selected books with enriched metadata:")
print("=" * 80)
df_selected[['pg_id', 'title', 'authors', 'author_birthdate', 'author_deathdate', 'wikipedia_url']].head(15)

In [ ]:
# Prepare data for temporal analysis
df_dated = df_selected[df_selected['author_deathdate'].notna()].copy()

# Convert to integer years
df_dated['death_year'] = pd.to_numeric(df_dated['author_deathdate'], errors='coerce')
df_dated = df_dated[df_dated['death_year'].notna()]

# Create century bins
df_dated['death_century'] = (df_dated['death_year'] // 100) + 1

print(f"TEMPORAL PROXY: Using author death dates (not publication dates)")
print(f"Books with extractable death years: {len(df_dated)} / {len(df_selected)}")
print(f"Date range: {df_dated['death_year'].min():.0f} - {df_dated['death_year'].max():.0f}")

In [ ]:
empty_author_year = print_empty_cells(df_selected, "author_deathdate", 10)

In [ ]:
# Visualization: Temporal distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Timeline scatter plot
ax1 = axes[0]
year_counts = df_dated['death_year'].value_counts().sort_index()
ax1.scatter(year_counts.index, year_counts.values, alpha=0.6, s=50, color='teal')
ax1.set_xlabel('Author death year')
ax1.set_ylabel('Number of texts')
ax1.set_title('Temporal distribution (Proxy: Author death dates)')
ax1.axvline(x=1500, 
            color='red', 
            linestyle='--', 
            alpha=0.5, 
            label='~Start of modernity')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Histogram by century
ax2 = axes[1]
century_counts = df_dated['death_century'].value_counts().sort_index()
century_labels = [f"{int((c-1)*100)}s" if c > 0 else f"{int(abs((c-1)*100))}s BCE" for c in century_counts.index]

ax2.bar(range(len(century_counts)), century_counts.values, color='teal')
ax2.set_xticks(range(len(century_counts)))
ax2.set_xticklabels(century_labels, rotation=45)
ax2.set_xlabel('Century (based on author death)')
ax2.set_ylabel('Number of texts')
ax2.set_title('Texts by century')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb01-temporal_distribution_proxy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary by period
print("\nTexts by period (death date proxy):")
print("=" * 60)
for century, count in century_counts.items():
    century_name = f"{int((century-1)*100)}s" if century > 0 else f"{int(abs((century-1)*100))}s BCE"
    percentage = count / len(df_dated) * 100
    bar = '█' * (count // 5)
    print(f"  {century_name:15} ({count:3}, {percentage:5.1f}%): {bar}")

# Critical thinking
- Do we have sufficient coverage for studying the transition to modernity?
- Which periods are under-represented?
- Can we answer our research questions with this temporal distribution?

 =============================================== YOUR THOUGHTS HERE ===============================================



---

# Part 9: Create manual lookup file

# Fill the gap

### _Complete additional metadata manually_

**Instructions:**
1. Create a CSV template with book information and Wikipedia URLs
2. Divide the list among group members
3. For each assigned book, research and record:
   - Original publication year
   - Source of information (Wikipedia, library catalog, etc.)
   - Confidence level (certain/uncertain/unknown)
   - Add the wikipedia urls and the authors' dates from the previous tasks
4. After completion, merge individual CSVs and import back into analysis

**Expected time:** ~30-60 minutes for 50 books

In [ ]:
# Create template for manual date lookup
manual_lookup = df_selected[[
    'pg_id', 'title', 'authors', 'wikipedia_url', 
    'author_birthdate', 'author_deathdate'
]].copy()

# Add empty columns for students to fill
manual_lookup['publication_year'] = ''  # To be filled manually
manual_lookup['year_source'] = ''       # Record the source (Wikipedia, library catalog, etc.)
manual_lookup['confidence'] = ''        # certain / uncertain / unknown
manual_lookup['notes'] = ''             # Any relevant notes

# Save template
template_path = OUTPUT_DIR / 'tables' / 'nb01-manual-enrichments.csv'
manual_lookup.to_csv(template_path, index=False)

print(f"✓ Manual date lookup template created: {template_path}")
print(f"  Total books to research: {len(manual_lookup)}")
print(f"  Books with Wikipedia URLs: {manual_lookup['wikipedia_url'].notna().sum()}")
print(f"\nNext steps:")
print(f"  1. Use the shared CSV file")
print(f"  2. Divide books among group members")
print(f"  3. Research each book and fill in: publication_year, year_source, confidence")
print(f"  4. Fill in missing data in wikipedia_url and author's deathdates")

In [ ]:
# Show sample of template
print("Sample of manual enrichment file:")
print("=" * 80)
manual_lookup.head(10)

In [ ]:
# Save selected book IDs as JSON
selected_ids_path = OUTPUT_DIR / 'reports' / 'nb01-selected-ids.json'
with open(selected_ids_path, 'w') as f:
    json.dump(selected_ids, f, indent=2)

print(f"✓ Saved selected book IDs: {selected_ids_path}")
print(f"  Total IDs: {len(selected_ids)}")

In [ ]:
# Save complete metadata as CSV
metadata_path = OUTPUT_DIR / 'tables' / 'nb01-selected-metadata.csv'
df_selected_to_file = df_selected[['pg_id',
                                   'title',
                                   'language',
                                   'authors',
                                   'subjects',
                                   'bookshelves',
                                   'category',
                                   'primary_author',
                                   'wikipedia_url',
                                   'author_birthdate',
                                   'author_deathdate']].copy()
    
df_selected_to_file.to_csv(metadata_path, index=False)

print(f"Saved selected book metadata: {metadata_path}")
print(f"Columns: {', '.join(df_selected_to_file.columns)}")

In [ ]:
# Create corpus summary report
summary_path = OUTPUT_DIR / 'reports' / 'nb01-corpus-summary.txt'

with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("NLP 2026: PHILOSOPHY CORPUS SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    
    f.write(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    
    f.write("CORPUS SIZE\n")
    f.write("-" * 80 + "\n")
    f.write(f"Total books selected: {len(df_selected)}\n")
    f.write(f"Books with Wikipedia URLs: {df_selected['wikipedia_url'].notna().sum()}\n")
    f.write(f"Books with author death dates: {df_selected['author_deathdate'].notna().sum()}\n\n")
    
    f.write("SELECTION CRITERIA\n")
    f.write("-" * 80 + "\n")
    for key, value in SELECTION_CRITERIA.items():
        f.write(f"{key}: {value}\n")
    f.write("\n")
    
    f.write("LANGUAGE DISTRIBUTION\n")
    f.write("-" * 80 + "\n")
    lang_dist = df_selected['language'].value_counts()
    for lang, count in lang_dist.items():
        f.write(f"{lang}: {count} ({count/len(df_selected)*100:.1f}%)\n")
    f.write("\n")
    
    f.write("TOP 20 AUTHORS\n")
    f.write("-" * 80 + "\n")
    author_dist = df_selected['primary_author'].value_counts().head(20)
    for author, count in author_dist.items():
        f.write(f"{author}: {count} books\n")
    f.write("\n")
    
    if len(df_dated) > 0:
        f.write("TEMPORAL DISTRIBUTION (PROXY: AUTHOR DEATH DATES)\n")
        f.write("-" * 80 + "\n")
        f.write(f"Books with dates: {len(df_dated)} / {len(df_selected)}\n")
        f.write(f"Date range: {df_dated['death_year'].min():.0f} - {df_dated['death_year'].max():.0f}\n\n")
        
        for century, count in century_counts.items():
            century_name = f"{int((century-1)*100)}s" if century > 0 else f"{int(abs((century-1)*100))}s BCE"
            f.write(f"{century_name}: {count} books ({count/len(df_dated)*100:.1f}%)\n")
        f.write("\n")
    
    f.write("CRITICAL LIMITATIONS\n")
    f.write("-" * 80 + "\n")
    f.write("- English language bias (excludes non-translated works)\n")
    f.write("- Translation selection bias (who gets translated?)\n")
    f.write("- US copyright law bias (mostly pre-1928 publications)\n")
    f.write("- Digitization bias (what volunteers chose to digitize)\n")
    f.write("- Canon bias (over-representation of canonical figures)\n")
    f.write("- Temporal proxy: Using author death dates, NOT publication dates\n")
    f.write("\n")
    
    f.write("NEXT STEPS\n")
    f.write("-" * 80 + "\n")
    f.write("1. Complete manual publication date lookup\n")
    f.write("2. Import completed dates and update metadata\n")
    f.write("3. Proceed to Notebook 02: Text downloading and preprocessing\n")
    f.write("\n")
    
    f.write("OUTPUT FILES\n")
    f.write("-" * 80 + "\n")
    f.write(f"- {selected_ids_path.name}\n")
    f.write(f"- {metadata_path.name}\n")
    f.write(f"- {template_path.name}\n")
    f.write(f"- {summary_path.name}\n")

print(f"✓ Saved corpus summary: {summary_path}")

In [ ]:
# Display summary
with open(summary_path, 'r', encoding='utf-8') as f:
    print(f.read())

---
**What we have accomplished:**
1. ✓ Explored 70,000+ books in Project Gutenberg catalog
2. ✓ Identified and filtered philosophy books
3. ✓ Analyzed language, subject, and author distribution
4. ✓ Applied selection criteria to create curated corpus
5. ✓ Extracted Wikipedia URLs and author dates from RDF files
6. ✓ Created temporal analysis using author death dates (proxy)
7. ✓ Generated manual date lookup template for students
8. ✓ Saved all outputs for next steps

**Critical reflections:**
- We have documented biases in language, translation, digitization, and canon
- Author death dates provide rough periodization but NOT precise publication dates
- Manual date research is necessary for accurate temporal analysis

**Next notebook:**
- Notebook 02: Download and preprocess the selected texts
- This requires completion of manual date lookup (optional but recommended)

**Files created:**
- `nb01-selected-ids.json` - Book IDs to download
- `nb01-selected-metadata.csv` - Complete metadata
- `nb01-manual-date-lookup.csv` - Template for date research
- `nb01-corpus-summary.txt` - Human-readable summary
- Visualization PNGs in output directory

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A1a highlight;
```